# HOME
---
## **Introdução** 
O modelo de Longevidade foi desenvolvido com o objetivo de avaliar o risco de ocorrência de um evento de óbito entre indivíduos de idades avançadas (65 a 90 anos de idade) em um horizonte de curto/médio prazo (em até 6 meses). O output do modelo é um score/probabilidade de 0 a 1 associado à um nível de risco que assume valores de 1 a 6, onde os indivíduos alocados no nível 1 apresentam risco mínimo (conforme atribuído pelo modelo) de virem a óbito em um futuro próximo enquanto aqueles alocados no nível 6 são os que o modelo julga mais vúlneráveis ao acontecimento. A utilização desse dado no contexto de Crédito no Banco Itaú-Unibanco visa mitigar o risco representado por aqueles clientes que vêm a faltar no cumprimento de suas obrigações financeiras em decorrência do evento de óbito, tornando-se mais um insumo disponível no desenvolvimento de políticas e processos de análise de risco de crédito cada vez mais assertivos. Do momento da escrita desta documentação (Julho/2026) o Modelo de Longevidade já teve sua aplicação e impacto avaliados no contexto da política de concessão de Crédito Consignado para o segmento INSS enquanto estudos de possíveis aplicações tanto na camada Polaris do Modelo de Risco de Crédito Visão Cliente quanto no futuro Gene do Aposentado seguem em processo de avaliação.

## **Papéis e responsáveis**


# DEFINIÇÃO DE PÚBLICO
---
## **ESTUDO DA POPULAÇÃO**
Como passo inicial na determinação do público de modelagem, foi realizada uma caracterização da população a nível Bureau/Brasil. Usando como origem  o Book de Público (database.table), foi selecionada uma janela temporal extensa, abrangendo os meses de 05/2024 a 09/2025, dentro da qual realizou-se a avaliação de diferentes segmentações desses indivíduos que permitiram traçar um perfil de volumetria total e de distribuição do público dentro do domínio de cada segmentação. 

Conforme é possível ver no quadro abaixo, a média mensal de CPFs únicos fica praticamente constante ao longo de todas as referências analisadas. Duas marcações de segmentação do público que foram bastante importantes na jornada de desenvolvimento do modelo foram a de indivíduo com conta ativa no Banco (correntista) vs o oposto (os não correntistas), e a de contratação do produto de Crédito Consignado. Na figura abaixo é possível verificar a distribuição da população brasileira como um todo ao longo da janela de tempo analisada bem como dentro das segmentações definidas. Para cada um dos grupos é trazido o valor da taxa de óbito em até 6 meses.

[TABELA EXCEL VOLUMETRIA POPULAÇÃO]

## **AMOSTRAGEM DO PÚBLICO DE DESNVOLVIMENTO**
Tendo à disposição as análises da população deu-se segmento ao processo de amostragem para obtenção do conjunto de dados de desenvolvimento. Foi definida a volumetria total de 1 milhão de indivíduos para essa amostra, garantindo assim um volume suficiente de dados para a aplicação do processo de modelagem. A amostra foi gerada buscando uma porporção de 50% correntistas e 50% não correntistas, intencionalmente aumentando a proporção amostral dos indivíduos para os quais o Banco possui mais informação disponível. Além disso, dentro do público correntista foram selecionados 250 mil indivíduos que adquiriram um contrato de Crédito Consignado ao longo das safras analisadas para permitir a avaliação do Modelo de Longevidade aplicado a concessão de um produto de crédito. Dentro de cada um dos grupos definidos a amostragem foi realizada de forma a espelhar a taxa de óbito observada por referência na população total.

[TABELA EXCEL VOLUMETRIA AMOSTRA]

## **HISTÓRICO DE DESENVOLVIMENTO**
O histórico de desenvolvimento do Modelo de Longevidade abrangeu os meses de Maio de 2024 a Setembro de 2025 (17 safras). A definição dessa janela levou em consideração a recência dos dados, a disponibilidade histórica de informações proveninetes de algumas fontes de dados e também a definição da variável resposta do modelo (link TARGET) que requer um período de 6 meses de maturação para determinação. Desse histórico, os 13 primeiros meses foram utilizados no treinamento do modelo enquanto os 4 meses finais foram aplicados no teste (out of time). É o conjnto de dados de teste que permite validar a capacidade final de generalização do modelo após seu desenvolvimento.


# VARIÁVEL RESPOSTA
---
## **MARCAÇÃO DO EVENTO DE ÓBITO**
Para marcação do evento de óbito na base de desenvolvimento foi utilizado o Book de Público (database.table). O Book disponibiliza um campo que traz informação sobre o status atual do indivíduo perante a Receita Federal e, dentre os valores assumidos pelo mesmo, o critério 'status_rct_federal = 3' marca os registros para os quais o órgão possui apontamento cadastral de óbito. 

Após realizada a marcação inicial, foram verificadas algumas inconsistências no conjunto de dados obtido, tais quais indivíduos que apresentavam registro de óbito em mais de uma data e indivíduos que apresentavam status de CPF ativo mesmo após uma marcação de evento de óbito em data anterior. Para tratar esses casos de incosistência foi adotada a regra de utilizar sempre a data da marcação mais antiga de óbito (ou a primeira) como padrão.

O procedimento completo de marcação do evento de óbito a partir das fontes de informação originais conforme realizado durante o processo de desenvolvimento do modelo pode ser consultado no seguinte Notebook[NOTEBOOK]

## **DEFINIÇÃO DO TARGET DO MODELO**
A definição da variável resposta do modelo buscou aproximar a tarefa de predição de óbito àquela de estimação de risco de crédito, permitindo a aplicação do mesmo arcabouço de métodos e ferramentas já bastante familiares aos cientistas de dados do Crédito PF à resolução do problema. Para tal, foi definida uma janela de tempo em meses (análoga a janela de performance para o risco de crédito) e a variável resposta do modelo definida de forma a assumir o valor 1 em caso de evento de óbito observado na janela a paritr da data de referência de entrada do indivíduo na base de dados ou o valor 0 no caso contrário. 

Foram realizados alguns experimentos para definição da quantidade exata de meses utilizados na composição janela de tempo de inferência do modelo. Conforme exposto na Figura abaixo, analisou-se a variação da taxa de óbito observada para diferentes faixas etárias ao longo de janelas abrangendo diferentes quantidades de meses. Observa-se que não foi constatada nenhuma tendência de estabilização da taxa de óbito conforme o aumento do número de meses, portanto a variável resposta foi definida com base na janela de performance na qual deseja-se discriminar o evento. No caso do Modelo de Longevidade decidiu-se então adotar a janela de 6 meses com a variável resposta do modelo assumindo o conceito de: ocorrência de óbito nos 6 meses seguintes a referência de entrada do indivíduo na base de dados.

[TABELA/GRÁFICOS ESTUDO EXCEL JANELA]

## **CARACTERIZAÇÃO DO EVENTO**


# EXPLORAÇÃO DE DADOS
---
## **PREMISSAS**
Durante o processo de Discovery do projeto foi realizada uma revisão da literatura científica existente relacionada ao mesmo domínio de problema: desenvolvimento de modelos de Machine Learning para predição de evento de óbito. Os trabalhos analisados evidenciam uma diferença significativa nos patamares de discriminação obtidos por modelos que utilizam variáveis com indicativos diretos de saúde dos indivíduos (resultados de exames laboratoriais, diagnósticos de doenças pré existentes...) quando comparados a modelos treinados sem este tipo específico de informaçãp. Tendo em vista esse fato e sabendo que os dados disponíveis para o desenvolvimento do Modelo de Longevidade no contexto do Banco Itaú-Unibanco não incluem este tipo de dado, decidiu-se direcionar a exploração das variáveis explicativas para 3 grandes grupos de dados:
    
    - Essenciais/Demográficos: que caracterizam os indivíduos de acordo com sua identidade e ambiente;
    - Padrões de consumo: tendências (e outras estatísticas) de gastos em determinadas categorias;
    - Proxies de fragilidade: indicativos indiretos de piora no estado de saúde dos indivíduos;

[LINK EXCEL REVIEW LITERATURA]

## **FONTES DE INFORMAÇÕES EXPLORADAS**
Abaixo encontram-se indicados todos os conjuntos de dados que foram avaliados no processo de definição das variáveis explicativas do modelo. Para cada um é fornecida uma breve descrição além do caminho da tabela de origem para consulta.

[TABELA DE FONTES DE INFORMAÇÃO]

## **SELEÇÃO DE VARIÁVEIS**

### - ANÁLISE DE PREENCHIMENTO
### - ANÁLISE DE VARIABILIDADE
### - ANÁLISE DE CORRELAÇÃO
### - RECURSIVE FEATURE ELIMINATION


# DESENVOLVIMENTO DO MODELO
---
## **OTIMIZAÇÃO DE HIPERPARÂMETROS**
O processo de otimização de hiperparâmetros utilizou o Optuna[LINK] como ferramenta. Abaixo encontram-se listados os parâmetros de entrada utilizados no processo de otimização bem como os hiperparâmetros finais selecionados, conforme pode-se consultar acessando o seguinte Notebook[NOTEBOOK].

Para obtenção de estimativas mais robustas de performance do modelo, o processo de otimização realiza uma validação cruzada no intuito de avaliar a métrica de performance em diferentes quebras do conjunto de dados de treinamento para cada combinação diferente de hiperparâmetros testados. Vale ressaltar que, por questões da dinâmica temporal inerente ao problema modelado e no intuito de evitar qualquer possibilidade de 'data leakage', os folds do processo de validação cruzada foram definidos de maneira a garantir que os dados utilizados para treinamento do modelo fossem anteriores aos dados utilizados na validação, em um padrão de janelas temporais crescentes.

## **TREINAMENTO DO MODELO**
Foi treinado um modelo LGBM (light gradient boosting machine) [LINK LGBM] utilizando os hiperparâmetros ótimos selecionados na etapa descrita acima. O quadro abaixo exibe as versões de cada biblioteca Python utilizada para o treinamento do modelo.

[VERSÕES DAS LIBS PYTHON]

Na figura abaixo é possível visualizar as curvas de aprendizado obtidas durante o treinamento do modelo tanto para o conjunto de dados de treinamento quanto para o de validação. 

## **AVALIAÇÃO DO MODELO NA BASE DE DESENVOLVIMENTO**

[GINIS]

[FEATURE IMPORTANCES]

[ANÁLISE DE DECIS]


## **AVALIAÇÃO DE MODELAGEM POR SUBPOP**


# DEFINIÇÃO DOS NÍVEIS DE RISCO
---
## **OBTENÇÃO DOS PONTOS DE CORTE**
## **CARACTERIZAÇÃO DOS NÍVEIS DE RISCO**
## **AVALIANDO CAPACIDADE DE ORDENAÇÃO**


# VALIDAÇÃO DO MODELO E RESULTADOS
---
## **BENCHMARKING DE VALIDAÇÃO**
## **ANÁLISES DE EXPLICABILIDADE**
## **AVALIAÇÃO NO CONTEXTO DO CONSIGNADO INSS**

# CONTROLES OBRIGATÓRIOS
---



# EXPLORAÇÃO DE DADOS
---
## **PREMISSAS**
Durante o processo de Discovery do projeto foi realizada uma revisão da literatura científica existente relacionada ao mesmo domínio de problema: desenvolvimento de modelos de Machine Learning para predição de evento de óbito. Os trabalhos analisados evidenciam uma diferença significativa nos patamares de discriminação obtidos por modelos que utilizam variáveis com indicativos diretos de saúde dos indivíduos (resultados de exames laboratoriais, diagnósticos de doenças pré existentes...) quando comparados a modelos treinados sem este tipo específico de informaçãp. Tendo em vista esse fato e sabendo que os dados disponíveis para o desenvolvimento do Modelo de Longevidade no contexto do Banco Itaú-Unibanco não incluem este tipo de dado, decidiu-se direcionar a exploração das variáveis explicativas para 3 grandes grupos de dados:
    
    - Essenciais/Demográficos: que caracterizam os indivíduos de acordo com sua identidade e ambiente;
    - Padrões de consumo: tendências (e outras estatísticas) de gastos em determinadas categorias;
    - Proxies de fragilidade: indicativos indiretos de piora no estado de saúde dos indivíduos;

[LINK EXCEL REVIEW LITERATURA]

## **FONTES DE INFORMAÇÕES EXPLORADAS**
Abaixo encontram-se indicados todos os conjuntos de dados que foram avaliados no processo de definição das variáveis explicativas do modelo. Para cada um dos conjuntos é indicado o tipo de informação contido na tabela de origem, o caminho da tabela de origem para consulta e o domínio no qual se encaixam segundo as 3 divisões definidas no tópico anterior. Além disso, foi incluída uma breve justificativa para os casos nos quais o conjunto dados não avançou da etapa de exploração para a fase seguinte de seleção de variáveis.

[TABELA DE FONTES DE INFORMAÇÃO]

[ACRESCENTAR SEÇÃO DE FEATURE ENGINEERING?]

Para prosseguir com o processo de modelagem, as fontes de informação aprovadas para seguir na sequência do processo de modelagem foram cruzadas com o público de desenvolvimento obtido conforme a descrição detalhada na seção [Deifnição de Públio](<!--LINK-->). O processo de extração de dados e cruzamento com a tabela de público pode ser consultado através do Notebook [1 - Featue extraction](<!--LINK-->).  

## **SELEÇÃO DE VARIÁVEIS**
O dataset utilizado como input para o processo de Seleção de Variáveis é composto de 746.029 linhas (volumetria da base final amostrada do público de desenvolvimento) e 1618 colunas compreendendo 1572 colunas de variáveis canditatas e 46 colunas que emglobam diferentes características úteis para segmentação e análise dos resultados além de colunas de identificação dos registros e da variável resposta. O objetivo final deste processo é a seleção de um subconjunto das 1572 variáveis candidatas garantindo que as features selecionadas forneçam ao modelo treinado informação de qualidade para inferir a variável resposta a partir dos dados  enquanto concomitantemente atendendo a certos requisitos necessários ou desejáveis à sua aplicação no treinamento do modelo final.

Devido ao papel central que a informação a respeito da idade ocupa no domínio do problema de predição de óbito, o processo de Seleção de Variáveis foi aplicado adotanto uma estratégia de coortes: o público amostrado de desenvolvimento foi dividido em 8 grupos de acordo com a segmentação correntista/não correntista e a sua faixa etária (4 quebras, abrangendo uma faixa de 5 anos cada) e cada um dos critérios do pipeline de Seleção de variáveis foi aplicado grupo a grupo individualmente. Na avaliação final de cada critério as variáveis foram mantidas no conjunto de candidatas somente quando verificado o preenchimento dos requisitos mínimos para todos os 8 grupos ao mesmo tempo.

Abaixo encontra-se descrita cada etapa do pipeline de Seleção de Variáveis aplicado no desenvolvimento do Modelo de Longevidade seguindo a ordem de aplicação.

### - AVALIAÇÃO DE PREENCHIMENTO HISTÓRICO
Pré requisito: dados disponíveis em todas as referências temporais do público de desenvolvimento, sem a existência de falhas de preenchimento em alguma safra específica

### - AVALIAÇÃO DE ESTABILIDADE TEMPORAL (IEP) 
Pré requisito: média dos valores de IEP (Índice de Estabilidade Populacional) calculado safra a safra (referentes a safra inicial) inferior a 10%

### - AVALIAÇÃO DE PREENCHIMENTO MÍNIMO
Pré requisito: percentual de valores preenchidos na coluna superior a 1%

### - AVALIAÇÃO DE VARIÂNCIA
Pré requisito: variância dos valores preenchidos na coluna superior a 0 (que os valores preenchidos não sejam todos iguais)

### - AVALIAÇÃO DE CORRELAÇÃO [^1]
Pré requisito: correlação de Spearman (em valor absoluto) entre duas variáveis inferior a 0,7 

### - DECISÃO JULGAMENTAL
Pré requisito: avaliação caso a caso, mas geralmente descartadas devido a baixa relação com o domínio do problema ou identificação de alguma característica indesejada após avaliação visual da distribuição das variáveis via ferramenta [Pytinela](<!--LINK-->) ([resultados](<!--LINK-->)) 

### - RECURSIVE FEATURE ELIMINATION (COM VALIDAÇÃO CRUZADA)
Pré requisito: seleciona o conjunto de variáveis para o qual o valor médio da métrica de performance atinge seu máximo (média sobre os dados de validação nos diferentes folds)


A Figura X mostra a estrutura do pipeline de Seleção de Variáveis aplicado evidenciando a quantidade de features remanescentes após cada etapa.

<!--FIGURA-->


## **VARIÁVEIS FINALISTAS DO MODELO**
A tabela a seguir traz a listagem das variáveis finalistas do modelo acompanhada de uma breve descrição:


[^1]: Durante a avaliação de correlação as variáveis são analisadas par a par e é necessário decidir qual das 2 variáveis do par deve ser descartada quando o valor aferido de correlação ultrapassa o critério definido. No contexto do Modelo de Longevidade foi desenvolvido um "score de qualidade" geral das variáveis que foi calculado previamente à aplicação da avaliação de correlação permitindo ordená-las relativamente umas as outras e descartar a de menor valor quando comparadas par a par. Esse "score de qualidade" é uma métrica que avalia as features univariadamente de acordo com cinco quesitos: poder preditivo (média do gini ao longo do tempo), estabilidade (IEP máximo ao longo do tempo), estabilidade da relação preditiva (desvio padrão do gini ao longo do tempo), robustez da relação preditiva (média da correlação de postos de Spearman ao longo do tempo) e a robustez preditiva nos coortes (gini do pior grupo). Um peso é atribuído a cada um dos quesitos para geração de um score final. A definição da função de escoragem pode ser consultada [AQUI](<!--LINK utils.py-->). 



# DESENVOLVIMENTO DO MODELO
---
## **PROCESSO DE MODELAGEM ITERATIVO**
Ao longo da execução de um projeto de Ciência de Dados é comum adotar um processo iterativo de desenvolvimento do modelo onde soluções intermediárias vão sendo produzidas e cada nova versão traz melhorias ou corrige direcionamentos adotados em versões anteriores (framework CRISPR-DM, por exexmplo). No desenvolvimento do Modelo de Longevidade foram produzidas 4 grandes versões de modelo (majors) e a última delas é a que foi produtizada e sobre a qual discorre essa documentação. Abaixo encontram-se listados os principais fatores que diferenciam cada grande versão de suas anteriores:

<!--FIGURA-->


## **OTIMIZAÇÃO DE HIPERPARÂMETROS**
Tendo definido o conjunto final de variáveis para compor o modelo, conforme detalhado na seção [Exploração de Daods](<!--LINK-->), deu-se seguimento ao processo de modelagem através da etapa de otimização de hiperparâmetros. Abaixo encontram-se listados os parâmetros de entrada utilizados pela ferramenta [Optuna](<!--LINK-->) durante o processo de otimização bem como os hiperparâmetros finais selecionados. 

<!--TABELA-->

Para obtenção de estimativas mais robustas de performance do modelo, o processo de otimização realiza uma validação cruzada no intuito de avaliar a métrica de performance em diferentes quebras do conjunto de dados de treinamento para cada combinação diferente de hiperparâmetros testados. Vale ressaltar que, por questões da dinâmica temporal inerente ao problema modelado e no intuito de evitar qualquer possibilidade de 'data leakage', os folds do processo de validação cruzada foram definidos de maneira a garantir que os dados utilizados para treinamento do modelo fossem anteriores aos dados utilizados na validação, em um padrão de janelas temporais crescentes, conforme sugere a literatura do tema [^2] [^3]. Ainda que o conjunto de dados de treinamento tenha sido corretamente controlado via deduplicação dos registros para que cada indivíduo fosse representado apenas uma vez no conjunto de dados completo, mitigando desta maneira o possível risco de treinar o modelo em dados futuros, a manutenção da ordenação temporal dos folds durante o processo de validação cruzada permite mitigar riscos com potencial origem no shift da distribuição de dados ou no conceito de variáveis que deterioram a performance em um regime de avaliação futuro e que podem ser mascarados quando analisados sob um proceso de validação cruzada utilizando folds definidos aleatoriamente.

O processo completo de otimização e seus outputs podem ser consultados acessando o  Notebook [X - Modelagem](<!--LINK-->).

[^2]: [https://arxiv.org/abs/2112.10078](https://arxiv.org/abs/2112.10078)
[^3]: [https://papers.ssrn.com/sol3/papers.cfm?abstract_id=6336198](https://papers.ssrn.com/sol3/papers.cfm?abstract_id=6336198)


## **TREINAMENTO DO MODELO**
Foi treinado um modelo [LGBM](<!--LINK-->) (Light Gradient Boosting Machine) utilizando os hiperparâmetros ótimos selecionados na etapa descrita acima. Os quadros abaixo exibem as versões de cada biblioteca Python utilizada para o treinamento do modelo.

<!--VERSÕES DAS LIBS PYTHON-->

Durante o processo de treinamento foram utilizadas as safras de 04 e 05/2025 como conjunto de dados de validação além de definidas 2000 rodadas de boosting com um parâmetro de *early-stopping* de 50 rodadas a fim de mitigar possíveis riscos de overfitting. O modelo final treinado é um ensemble composto de 750 árvores e apresenta métricas de performance conforme o quadro abaixo: 

<!--GINIS-->

Na Figura X são exibidas as curvas de aprendizado obtidas durante o treinamento do modelo para as métricas de AUC (métrica de performance) e de *binary-logloss* (métrica de otimização da função de perda) tanto para o conjunto de dados de treinamento quanto para o de validação. As curvas permitem afirmar que, apesar do gap de performance do modelo observado entre os conjuntos de treino/validação/teste, o modelo demonstra um comportamento de aprendizado saudável, sem a presença de quaisquer sinais óbvios de overfit. O treinamento é interrompido na 750ª iteração quando ambas as curvas de treinamento e validação estão atingindo seu plateau e não observa-se aumento de performance no conjunto de treinamento após estagnação ou inversão da curva de validação conforme espera-se de regimes clássicos de overfit.  


## **AVALIAÇÃO DO MODELO NA BASE DE DESENVOLVIMENTO**

### - FEATURE IMPORTANCES

<!--FIGURA-->

### - PERFORMANCE DO MODELO

<!--FIGURA-->

### - DISCRIMINAÇÃO DO MODELO POR QUANTIL DO SCORE

<!--FIGURA-->


## **AVALIAÇÃO DE MODELAGEM POR SUBPOP**


# DEFINIÇÃO DOS NÍVEIS DE RISCO
---
## **OBTENÇÃO DOS PONTOS DE CORTE**
Os pontos de corte utilizados para determinação dos grupos de risco do Modelo de Longevidade foram obtidos através de um processo de otimização com o objetivo transformar o score contínuo do modelo de risco em um conjunto reduzido de grupos ordenados, contíguos e temporalmente estáveis. A descrição completa da metodologia aplicada pode ser consultada acessando sua [Documentação]<!--LINK-->.

A solução foi implementada através da função *optimal_binning_with_prebins* que pode ser consultada acessando o arquivo [utils.py](<!--LINK-->). Esta implementação combina quatro decisões centrais:

- Pré-binning ponderado, para representar a população e reduzir o domínio de busca.
- Minimização do IEP dos níveis de risco ao longo do tempo como objetivo primário, para alinhar a otimização à métrica de estabilidade desejada.
- Otimização lexicográfica, que define a minimização do IEP como objetivo primário e a estabilidade das taxas de evento de óbito como objetivo secundário.
- Programação dinâmica como técnica de otimização, para preservar a dependência monotônica e alcançar o ótimo global no espaço discretizado dos pré-bins.

A adoção desta metodologia e a implementação específica utilizada seguindo os critérios acima produz uma solução mais consistente do ponto de vista estatístico, computacional e de governança do que abordagens baseadas em cortes manuais, algoritmos gulosos (exemplo do Auto GH) ou programação dinâmica com estados excessivamente comprimidos.

## **CARACTERIZAÇÃO DOS NÍVEIS DE RISCO**

### - DISTRIBUIÇÃO DOS NÍVIES DE RISCO AO LONGO DO TEMPO

### - IEP AO LONGO DO TEMPO

### - TAXA DE ÓBITO POR NÍVEL DE RISCO AO LONGO DO TEMPO


# DISCUSSÕES E RESULTADOS DE APLICAÇÃO DO MODELO
---
## **BENCHMARKING DE PERFORMANCE**
## **AVALIAÇÃO NO CONTEXTO DO CONSIGNADO INSS**

# CONTROLES OBRIGATÓRIOS
---

# EXTRAÇÃO DE DADOS
---
## Base de Público

### **Público Alvo** 
O modelo tem como público alvo da escoragem os especialistas do produto de Microcrédito, que consistem em PJs que atuam no papel de intermedidores entre o banco e o cliente final na venda do produto. Estes PJs possuem cadastro junto a operação de Microcrédito e
seu rating/classificação mensal interfere seja na variabilidade de valores e prazos das ofertas que ele tem disponível para venda no mês vigente, seja na sua remuneração direta. Um breve acompanhamento da evolução da volumetria total mensal de especialistas ativos cadastrados junto a operação pode ser observada no gráfico abaixo:

(GRÁFICO VOLUMETRIA ESPEC/SSAFRA)

### **Histórico de Desenvolvimento** 

Para desenvolvimento e treino do modelo foram utilizadas as safras de contratação de julho/2023 a janeiro/2024. Já para avaliação do desempenho do modelo, foram utilizadas as safras de fevereiro/2024 e março/2024. A quantidade reduzida de referências utilizadas no processo de desenvolvimento do modelo se deve, entre outros, ao pequeno volume de dados existentes, consequência do fato de a própria operação de Microcrédito não possuir um número muito grande de especialistas cadastrados (qnd comparado a patamares que costumamos associar ao tamanho de conjuntos de dados de treinamento/teste de modelos de machine learning). Além disso, outros 2 fatores que contribuíram para o numero reduzido de referências utilizadas foram a disponibilidade do histórico de dados (não possuíamos histórico prévio a 2022) e a definição do target do modelo que, como tratado na seção específica sobre o tema (link Target), necessita acompanhar uma métrica de performance por no mínimo 3 meses a partir da referência avaliada para conseguir definir o label da variável resposta

### **Fontes de Informação** 

O público é construído de forma simples, utilizando a tabela de Gestão do Parceiro (db_source_repositoriosdedados_microcreditoanalytics_spec_01.tbma4_gestao_parceiro) onde são disponibilizadas as informações mais recentes e completas dos especialistas cadastrados junto a operação. 

Uma peculiaridade do processo é o fato de que, em alguns casos, mais de uma pessoa acaba atuando como especialista, porém atrelado ao mesmo CNPJ de cadastro. A esta configuração, deu-se o nome de PJ Ampliada, pelo fato do identificador único corresponder a um grupo de pessoas. Portanto, aqui torna-se importante salientar que, mesmo com a existëncia da configuração de PJ Ampliada, o rating do modelo é atribuído a nível de CNPJ, fazendo com que todos os subvendedores de uma mesma PJ Ampliada contribuam para o valor final obtido e que a todos seja atribuído o mesmo rating, tratando-os como uma única entidade. A chave única do púbico do modelo acaba sendo, então, o CNPJ/IDEQ3 do Especialista (ou grupo de subvendedores) e a safra a qual as demais informações do registro se referem.

### - Público Final de Desenvolvimento (amostragem)


## Variável Resposta
Para definição do conceito e marcação do **target** (ou variável resposta) utilizado no processo de modelagem do Angstrom foi feito uso da classificação pré existente do especialista. 

[Regras atribuição classificação especialista](../data/0_artifacts/target_dev_deals_orig.png) 

A tabela acima ilustra as regras utilizadas nesta classificação. Como é possível observar, os especialistas eram classificados em uma dentre as 7 categorias existentes e os critérios utilizados para tal se concentravam em refletir a inadimplência da sua carteira de clientes no curto prazo (máximo 3 meses anteriores, usando NPL e FPD). O rating do especialista, portanto, tratava-se de uma avaliação pós fato destes indicadores.

Com o objetivo de trazer um viés preditivo ao modelo desenvolvido, decidimos tentar abordar variáveis que expandissem nosso entendimento do especialista em si. Propusemos features para o modelo que procurassem refletir a qualidade da gestão que o especialista faz da própria carteira, bem como alguns dados sobre outras atividades profissionais exercidas por ele, seja na mesma área de atuação (Microcrédito) seja como sócio/proprietário de alguma PJ em qualquer outra área, e também alguns dados demográficos que pudessem caracterizar seu contexto socioeconômico.

Tomamos a decisão de modelar a probabilidade de o especialista vir a assumir o rating 'ddd' (pior rating da classificação anterior) em uma janela de até 3 meses. Temos, portanto:

... ==> target = 0
... ==> target = 1

Para marcação do target foi utilizada a base de classificacao dos especialistas, cruzada com a base de público


para utilização no presente modelo foi a performance 90 em 12 (atraso de 90 dias ou mais em um período de 12 meses). Para determinação da contratação interna de crédito imobiliário foi utilizado como indicador a presença de um número de contrato atrelado à proposta na base de propostas do produto disponível no SAS. O cruzamento da base de propostas com a base de contratação auxiliou na validação da informação. Para determinação da contratação de crédito imobiliário no mercado foram utilizadas as bases de [Contratação](https://confluence- itau.tecnologia.prod.ops.aws.cloud.ihf/x/kmP5Lw), (Performance Interna] (https://confluence-itau.tecnologia.prod.ops.aws.cloud.ihf/x/WWP5Lw) e  do projeto [Foucault] (https://confluence-itau.tecnologia.prod.ops.aws.cloud.ihf/pages/viewpage.action?pageId=799108541) que consolidam as informações contidas nas principais bases de público e performance (interna e mercado) que possuímos à disposição.


### - Bad Rate

## Variáveis explicativas
### - Fontes de informação exploradas
### - Pré seleção de variáveis


### - Extração de variáveis de books
Algumas variáveis mapeadas na listagem acima encontravam-se já disponíveis nos books de variáveis da NPM e, portanto, prontas para consumo em ambiente produtivo. Para estas variáveis, bastou a extração dos valores referentes aos registros da base de público, que foi efetuada através de uma query SQL de LEFT JOIN conforme pode ser visto nos arquivos X e Y. As variáveis que se encaixam nessa categoria são aquelas que na Tabela 1 possuem o valor "Extração Book NPM" na coluna "Método de extração". 

### - Síntese de variáveis a partir de outras fontes
Para a obtenção de algumas outras variáveis que decidimos utilizar na caracterização do especialista foi necessária a aplicação de algumas etapas de pré processamento antes de podermos usá-las efetivamente na modelagem. Nestes casos, ou o conceito que gostríamos de capturar teve de ser formulado através da agregação de algum dado mais bruto contido em outra tabela, ou o conceito já encontrava-se pronto porém nosso interesse era analisá-lo ao longo do tempo (usando janelas móveis, por exemplo). As variáveis que se encaixam nessa categoria são aquelas que na Tabela 1 possuem o valor "Variável Agregada" na coluna "Método de extração". Para todas estas variáveis é possível consultar AQUI as queries utilizadas em sua criação durante o desenvolvimento, e AQUI as queries que as reproduzem no processo batch de escoragem mensal. 

### - Tratamento de variáveis categóricas
Algumas variáveis categóricas tiveram de ser transformadas em numéricas para darmos cotinuidade ao processo de modelagem. Ainda que os algoritmos que envolvam árvores de decisão e seus derivados (como é o caso do Random Forest, utilizado na modelagem do Angstrom) consigam lidar bem com variáveis categóricas em sua maioria, demos preferência por trata-las manual e individualmente por serem poucas em quantidade e por termos maior controle sobre os possíveis valores finais assumidos pela feature.  


### - Análise de Variance Threshold
Foi realizado um corte das variáveis cuja distribuição de valores apresentavam pouca variância e que, portanto, não acrescentavam informação de grande valor ao modelo em desenvolvimento. Após realizar uma normalização de cada uma das features do conjunto de dados, utilizamos o VarianceThreshold da biblioteca Python scikit-learn e eliminamos as variáveis com variância inferior ao threshold definido de 0.02.


### - Análise visual da distribuição das features
Utilizamos o script do [Pytinela](), que acompanha a estabilidade populacional de uma feature e sua distribuição ao longo das diferentes safras analisadas. Variáveis que apresentam uma alta variabilidade populacional ao longo do tempo tendem a não ser muito indicadas para utilização em modelos preditivos: estes dependem da identificação de padrões nos dados de treino e posterior generalização do aprendizado para o conjunto de dados de teste. Caso a distribuição dos valores de uma feature varie de maneira muito brusca ou sem nenhum padrão discernível, ela pode acabar introduzindo "barulho" nos dados de treino ou até mesmo interagindo com alguma das outras variáveis preditivas, prejudicando a qualidade do modelo final. A partir desta análise acabamos excluíndo mais X variáveis do conjunto final.


### - Análise de correlação entre as variáveis explicativas
Uma outra análise que realizamos com o objetivo de selecionar as variáveis preditivas finais do modelo foi a avaliação do coeficiente de correlação de Pearson para cada par de variáveis candidatas. Os pares de variáveis que apresentaram valor do coeficiente de correlação maior que o threshold de 0.9 foram isolados e, dentre elas, foi selecionada a feature com maior valor de correlação com o target do modelo para permanecer no conjunto final. Nesta etapa, excluímos do conjunto final mais X variáveis, 
restando agora Y variáveis no total.


### - Variáveis finalistas para modelagem
Finalmente, concluídos os passos de pré seleção das variáveis explicativas, obtivemos um conjunto de XYZ variáveis totais, com as quais prosseguimos para a etapa seguinte do processo: a etapa de modelagem. Vale salientar que estas variáveis não necessariamente representam o conjunto de variáveis do modelo final, porém este é um subconjunto daquelas. Mais a frente, conforme detalhado na seção onde discutimos o processo de [Modelagem](), algumas variáveis deste conjunto acabam sendo descartadas após a avaliação do valor de Feature Importance que lhes é atribuído após o ajuste dos dados de treino ao modelo. 


### - Geração da Flat Table de Modelagem
Para as XYZ variáveis pré selecionadas, desenvolvemos um pipeline de extração e cruzamento com o público final para obtermos a flat table que serve de entrada para o processo de modelagem. O passo a passo do processo, que consiste em cruzamentos (JOINs SQL) da tabela de público com as devidas tabelas origem de cada dado, pode ser consultado [neste Jupyter Notebook](). 




# MODELAGEM
---
## Treinamento do modelo
### - Testes iniciais de modelagem
Com a [flat table de modelagem]() em mãos, realizamos alguns experimentos de modelagem utilizando métodos estatísticos e/ou algoritmos de Machine Learning que avaliamos pertinentes ao contexto do problema e ao conjunto de dados que tínhamos em mãos. 

Um dos primeiros testes que realizamos foi a utilização de um emsemble de árvores de decisão na tentativa de modelar os nossos dados. Mais especificamente, testamos a implementação da biblioteca Python [LightGBM](), em razão de já possuirmos bastante familiaridade com sua API e tmabém por já termos grande parte dos pipelines de treinamento e avaliação dos modelos criados a partir desta ferramenta bastante desenvolvidos e maduros em nossa equipe. Os experimentos, porém, não geraram resultados muito positivos: pudemos observar que, mesmo após o ajuste de hiperparâmetros, o modelo tendia a gerar um "overfit" bastante pronunciado dos dados de treino, conforme visto na curva de aprendizagem da Figura XX. A Figura mostra a evolução da função de custo nos dados de teste e validação do modelo após cada iteração do algoritmo (ou, neste caso, após a adição de cada árvore ao emsemble) e nos permite observar que, mesmo após a curva referente aos dados de validação já ter atingido seu mínimo, a curva dos dados de teste continua sua tendência decrescente indefinidamente. Este resultado nos mostra que a quantidade ideal de árvores no emsemble deveria ser YY (valor onde o mínimo da curva dos dados de validação é obtido) mas este valor além de se tratar de um número baixo de árvores também está muito distante do valor da função de custo obtida pelos dados de treinamento, nos levando a concluir, portante, que o algoritmo de Gradient Boosting é complexo demais (ou introduz muita variância) para modelar o nosso conjunto de dados que, como já falamos algumas vezes, possui poucas observações. Partimos então para experimentos com algoritmos mais simples, com menor quantidade de hiperparâmetros e complexidade no geral.

No outro extremo do espectro dos modelos de árvore, fomos testar as Árvores de Decisão simples. Construímos diversas árvores de decisão controlando, a princípio, a quantidade de níveis de profundidade ou a quantidade de folhas finais da árvore. Destes experimentos conseguimos observar que uma profundidade de 4 nívies, ou uma árvore com um máximo de 16 folhas, conseguia produzir valores de Gini que nos indicavam uma boa capacidade de discriminação (alto valor de Gini no conjunto de dados de validação) porém sem o "overfit" do conjunto de dos que observamos com o LGBM, evidenciado pela pequena diferença entre os valores de Gini para o conjunto de treino e de validação. Foram testadas algumas diferentes combinações de árvores e um apanhado dos resultados pode ser observado na Tabela XX.

Procurando um meio termo entre os emsembles gerados pelo método de Gradient Boosting (LGBM) e as Árvores de Decisão clássicas, decidimos testar o algoritmo de Random Forest, obtendo resultados bastante positivos desde o princípio. Alto poder de discriminação aliado a baixos indícios de "overfit" fizeram com a técnica acabasse sendo selecionada para gerar nosso modelo final. 



### - Seleção do modelo final
Tendo decidido pela utilização do algoritmo de Random Forest, partimos para as etapas finais de modelagem que consistem na otimização dos hiperparâmetros e na obtenção do modelo treinado. Utilizamos a ferramenta [Optuna]() para realizar o processo de otimização e os parâmetros obtidos para o treinamento da versão final do modelo estão expostos na Tabela XY a seguir.

TABELA XY

Vale destacar aqui alguns detalhes do processo de otimização. Primeiramente, o espaço amostral que definimos para exploração dos hiperparâmetros, que pode ser consultado na tabela XZ. A tabela contém o nome de cada um dos parâmetros do algoritmo Random Forest que selecionamos bem como o domínio de valores que definimos para cada um deles. O algoritmo de otimização realiza diversas iterações de amostragem destes parâmetros e para cada conjunto de valores amostrados é treinado um modelo e avaliada sua performance, na busca de maximizar (ou minimizar) uma determinada função objetivo.

Em segundo lugar, o algoritmo de amostragem escolhido, que foi o "Tree-structured Parzen Estimator" ou TPE. Este algoritmo é baseado em um processo de amostragem Bayesiano, onde a cada iteração (ou cada novo passo de exploração do espaço amostral de valores para os parâmetros do modelo) o algoritmo se utiliza das informações obtidas para realizar a escolha da amostra seguinte de maneira mais direcionada a atingir o objetivo de otimização. Se compararmos este funcionamento com o de um algoritmo de amostragem totalmente aleatório, onde a cada nova iteração seria selecionado um conjunto de valores ao acaso para utilização no treinamento do modelo, se torna mais evidente a motivação por trás desta escolha e as vantagens do algoritmo TPE.

O terceiro e último detalhe se refere a função objetivo que definimos como meta para otimização. Na grande maioria das vezes a função objetivo é definida a partir de uma métrica de performance do modelo. No nosso caso e no contexto geral de risco de crédito, normalmente estamos em busca dos parâmetros e do modelo capazes de maximizar o valor do Gini ou do KS quando avaliado no conjunto de dados de validação. Por conta da baixa volumetria de dados de treinamento que possuíamos e da forte tendência que os modelos testados apresentavam de "overfittar" nosso conjunto de dados, resolvemos definir uma função objetivo múltipla ou dupla no nosso caso: buscamos os valores de parâmetros que, ao mesmo tempo, fossem capazes de maximizar o valor de Gini porém minimizando a diferença de Gini observada entre o conjunto de dados de treinamento e de validação. Desta forma, nossa intenção é equilibrar da melhor maneira possível a qualidade/performance do modelo treinado com sua capacidade de generalização em dados desconhecidos quando em ambiente produtivo. 

Uma última etapa realizada para obtermos a versão treinada final do modelo foi a retirada do conjunto de treinamento das features que apresentavam valores de Feature Importance zerados, seguido de uma nova rodada de otimização de hiperparâmetros e retreino do modelo. No detalhe abaixo estão resumidos os dados referentes ao modleo Angstrom de acordo com a versão final implantada.  



### - Avaliação do modelo nos dados de desenvolvimento
Abaixo disponibilizamos alguns dados e métricas que costumamos analisar para aferição do poder discriminatório do modelo e que também servem, ainda que de maneira indireta, como indicativo da qualidade do modelo desenvolvido.  
        
        - Valor de gini          
        - Feature importances
        - Gráfico de discriminação do score
 
## Definição dos GHs
### - Obtenção dos grupos homogêneos
### - Distribuição do público nos GHs
### - GHs ordenando inadimplencia

## Validação e Resultados
### - Prevendo o especialista 'ddd'
### - Classificando o especialista 'hhh'
### - Matrizes de cruzamento e ginis

## Controles

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Iterable, Literal

import numpy as np
import pandas as pd


@dataclass(frozen=True)
class BinningSolution:
    """Resultado da otimização para uma quantidade específica de grupos."""

    n_groups: int
    cutpoints: list[float] | None
    population_psi: float | None
    bad_rate_instability: float | None
    feasible: bool


def _validate_inputs(
    df: pd.DataFrame,
    score_col: str,
    target_col: str,
    time_col: str,
    weight_col: str,
) -> pd.DataFrame:
    """Valida e prepara as colunas utilizadas na otimização."""

    required = [score_col, target_col, time_col, weight_col]
    missing = [col for col in required if col not in df.columns]

    if missing:
        raise KeyError(f"Colunas não encontradas: {missing}")

    data = df[required].copy()

    if data.isna().any().any():
        null_counts = data.isna().sum()
        null_counts = null_counts[null_counts > 0].to_dict()
        raise ValueError(
            "As colunas utilizadas não podem conter valores ausentes. "
            f"Valores ausentes encontrados: {null_counts}"
        )

    if not set(pd.unique(data[target_col])).issubset({0, 1}):
        raise ValueError(
            f"A coluna '{target_col}' deve ser binária, contendo apenas 0 e 1."
        )

    weights = data[weight_col].to_numpy(dtype=float)

    if not np.all(np.isfinite(weights)):
        raise ValueError("Os pesos devem ser finitos.")

    if np.any(weights <= 0):
        raise ValueError("Todos os pesos devem ser estritamente positivos.")

    scores = data[score_col].to_numpy(dtype=float)

    if not np.all(np.isfinite(scores)):
        raise ValueError("Os scores devem ser finitos.")

    return data


def _weighted_quantile_prebins(
    scores: np.ndarray,
    weights: np.ndarray,
    n_prebins: int,
) -> tuple[np.ndarray, np.ndarray]:
    """
    Constrói pré-bins aproximadamente equipopulosos na população ponderada.

    Scores iguais nunca são separados entre pré-bins.

    Returns
    -------
    prebin_ids:
        Identificador inteiro do pré-bin de cada observação.

    preliminary_cutpoints:
        Valores de score que separam os pré-bins.
    """

    if n_prebins < 2:
        raise ValueError("n_prebins deve ser maior ou igual a 2.")

    order = np.argsort(scores, kind="mergesort")
    sorted_scores = scores[order]
    sorted_weights = weights[order]

    unique_scores, first_indices = np.unique(
        sorted_scores,
        return_index=True,
    )

    weight_by_score = np.add.reduceat(
        sorted_weights,
        first_indices,
    )

    if unique_scores.size == 1:
        return np.zeros(scores.shape[0], dtype=int), np.array([], dtype=float)

    cumulative_weight = np.cumsum(weight_by_score)
    total_weight = cumulative_weight[-1]

    target_masses = (
        np.arange(1, n_prebins, dtype=float)
        / n_prebins
        * total_weight
    )

    candidate_indices = np.searchsorted(
        cumulative_weight,
        target_masses,
        side="left",
    )

    # O último score não pode ser um corte, pois criaria um pré-bin vazio.
    candidate_indices = candidate_indices[
        candidate_indices < unique_scores.size - 1
    ]

    candidate_indices = np.unique(candidate_indices)
    cutpoints = unique_scores[candidate_indices]

    # side="left": o próprio valor do corte permanece no pré-bin inferior.
    prebin_ids = np.searchsorted(
        cutpoints,
        scores,
        side="left",
    ).astype(int)

    return prebin_ids, cutpoints


def _lexicographic_better(
    candidate: tuple[float, float],
    incumbent: tuple[float, float] | None,
    tolerance: float = 1e-12,
) -> bool:
    """
    Compara custos lexicograficamente.

    Primeiro minimiza PSI. O custo de bad rate é utilizado somente como
    critério secundário.
    """

    if incumbent is None:
        return True

    candidate_psi, candidate_br = candidate
    incumbent_psi, incumbent_br = incumbent

    if candidate_psi < incumbent_psi - tolerance:
        return True

    if abs(candidate_psi - incumbent_psi) <= tolerance:
        return candidate_br < incumbent_br - tolerance

    return False


def optimal_binning_with_prebins(
    df: pd.DataFrame,
    score_col: str,
    target_col: str,
    time_col: str,
    weight_col: str,
    n_groups_list: Iterable[int],
    n_prebins: int = 100,
    risk_direction: Literal[
        "higher_score_higher_risk",
        "higher_score_lower_risk",
    ] = "higher_score_higher_risk",
) -> dict[int, BinningSolution]:
    """
    Determina cortes de score temporalmente estáveis em um domínio pré-binado.

    A solução é globalmente ótima no espaço discreto dos pré-bins para:

    1. a função objetivo aditiva definida nesta implementação;
    2. as restrições de contiguidade, cobertura, tamanho mínimo automático
       e monotonicidade entre grupos adjacentes;
    3. a otimização lexicográfica entre PSI e instabilidade de bad rate.

    Parameters
    ----------
    df:
        Base de desenvolvimento.

    score_col:
        Coluna contendo o score.

    target_col:
        Target binário, com 1 representando o evento.

    time_col:
        Referência temporal, por exemplo mês ou safra.

    weight_col:
        Peso amostral utilizado para reconstruir a população.

    n_groups_list:
        Quantidades de grupos a serem avaliadas.

    n_prebins:
        Número desejado de pré-bins ponderados. O número efetivo pode ser
        menor em razão de scores repetidos.

    risk_direction:
        Define se scores maiores representam maior ou menor risco.

    Returns
    -------
    dict[int, BinningSolution]
        Resultado para cada quantidade solicitada de grupos.
    """

    data = _validate_inputs(
        df=df,
        score_col=score_col,
        target_col=target_col,
        time_col=time_col,
        weight_col=weight_col,
    )

    requested_groups = sorted({int(k) for k in n_groups_list})

    if not requested_groups or requested_groups[0] < 2:
        raise ValueError(
            "n_groups_list deve conter inteiros maiores ou iguais a 2."
        )

    scores = data[score_col].to_numpy(dtype=float)
    targets = data[target_col].to_numpy(dtype=float)
    weights = data[weight_col].to_numpy(dtype=float)

    prebin_ids, _ = _weighted_quantile_prebins(
        scores=scores,
        weights=weights,
        n_prebins=n_prebins,
    )

    data["_prebin_id"] = prebin_ids
    data["_weighted_bad"] = (
        data[target_col].to_numpy(dtype=float)
        * data[weight_col].to_numpy(dtype=float)
    )

    # Garante IDs consecutivos mesmo quando quantis colapsam por empates.
    unique_prebins = np.sort(data["_prebin_id"].unique())
    remapping = {
        old_id: new_id
        for new_id, old_id in enumerate(unique_prebins)
    }

    data["_prebin_id"] = data["_prebin_id"].map(remapping).astype(int)

    bin_boundaries = (
        data.groupby("_prebin_id", sort=True)[score_col]
        .max()
        .to_numpy(dtype=float)
    )

    times = np.sort(data[time_col].unique())
    time_mapping = {
        time_value: index
        for index, time_value in enumerate(times)
    }

    grouped = (
        data.groupby(["_prebin_id", time_col], sort=True)
        .agg(
            weighted_population=(weight_col, "sum"),
            weighted_bads=("_weighted_bad", "sum"),
            sample_count=(target_col, "size"),
        )
        .reset_index()
    )

    n_bins = len(bin_boundaries)
    n_times = len(times)

    weighted_population = np.zeros((n_bins, n_times), dtype=float)
    weighted_bads = np.zeros((n_bins, n_times), dtype=float)
    sample_count = np.zeros((n_bins, n_times), dtype=float)

    for row in grouped.itertuples(index=False):
        bin_index = int(row._prebin_id)
        time_index = time_mapping[getattr(row, time_col)]

        weighted_population[bin_index, time_index] = (
            row.weighted_population
        )
        weighted_bads[bin_index, time_index] = row.weighted_bads
        sample_count[bin_index, time_index] = row.sample_count

    total_population_by_time = weighted_population.sum(axis=0)

    if np.any(total_population_by_time <= 0):
        raise ValueError(
            "Todos os períodos devem possuir massa populacional positiva."
        )

    cumulative_population = np.cumsum(weighted_population, axis=0)
    cumulative_bads = np.cumsum(weighted_bads, axis=0)
    cumulative_sample_count = np.cumsum(sample_count, axis=0)

    def interval_sum(
        cumulative_matrix: np.ndarray,
        start: int,
        end: int,
    ) -> np.ndarray:
        if start == 0:
            return cumulative_matrix[end].copy()

        return cumulative_matrix[end] - cumulative_matrix[start - 1]

    # Estatísticas de todos os segmentos possíveis.
    segment_population = np.zeros(
        (n_bins, n_bins, n_times),
        dtype=float,
    )
    segment_share = np.zeros_like(segment_population)
    segment_bad_rate = np.full_like(segment_population, np.nan)
    segment_sample_count = np.zeros_like(segment_population)

    segment_psi_cost = np.full(
        (n_bins, n_bins),
        np.inf,
        dtype=float,
    )
    segment_br_cost = np.full(
        (n_bins, n_bins),
        np.inf,
        dtype=float,
    )

    total_population_all_periods = total_population_by_time.sum()
    period_weights = (
        total_population_by_time / total_population_all_periods
    )

    epsilon = np.finfo(float).eps

    for start in range(n_bins):
        for end in range(start, n_bins):
            population_vector = interval_sum(
                cumulative_population,
                start,
                end,
            )
            bad_vector = interval_sum(
                cumulative_bads,
                start,
                end,
            )
            count_vector = interval_sum(
                cumulative_sample_count,
                start,
                end,
            )

            share_vector = (
                population_vector / total_population_by_time
            )

            bad_rate_vector = np.divide(
                bad_vector,
                population_vector,
                out=np.full(n_times, np.nan, dtype=float),
                where=population_vector > 0,
            )

            pooled_population = population_vector.sum()
            pooled_bad_rate = (
                bad_vector.sum() / pooled_population
                if pooled_population > 0
                else np.nan
            )

            reference_share = np.sum(
                period_weights * share_vector
            )

            # Contribuição aditiva do segmento ao PSI médio temporal.
            psi_contributions = (
                share_vector - reference_share
            ) * np.log(
                (share_vector + epsilon)
                / (reference_share + epsilon)
            )

            population_psi = float(
                np.sum(period_weights * psi_contributions)
            )

            # Instabilidade direta da taxa de evento em torno da taxa pooled.
            valid_br = np.isfinite(bad_rate_vector)

            if np.any(valid_br):
                normalized_period_weights = period_weights[valid_br]
                normalized_period_weights = (
                    normalized_period_weights
                    / normalized_period_weights.sum()
                )

                br_instability = float(
                    np.sum(
                        normalized_period_weights
                        * (
                            bad_rate_vector[valid_br]
                            - pooled_bad_rate
                        )
                        ** 2
                    )
                )
            else:
                br_instability = np.inf

            segment_population[start, end] = population_vector
            segment_share[start, end] = share_vector
            segment_bad_rate[start, end] = bad_rate_vector
            segment_sample_count[start, end] = count_vector
            segment_psi_cost[start, end] = population_psi
            segment_br_cost[start, end] = br_instability

    results: dict[int, BinningSolution] = {}

    for n_groups in requested_groups:
        if n_groups > n_bins:
            results[n_groups] = BinningSolution(
                n_groups=n_groups,
                cutpoints=None,
                population_psi=None,
                bad_rate_instability=None,
                feasible=False,
            )
            continue

        # Regra interna: cada grupo deve conter, em cada período,
        # ao menos metade da participação esperada em grupos equipopulosos.
        automatic_minimum_share = 0.5 / n_groups

        feasible_segment = np.zeros(
            (n_bins, n_bins),
            dtype=bool,
        )

        for start in range(n_bins):
            for end in range(start, n_bins):
                share_vector = segment_share[start, end]
                counts = segment_sample_count[start, end]
                bad_rates = segment_bad_rate[start, end]

                feasible_segment[start, end] = bool(
                    np.all(share_vector >= automatic_minimum_share)
                    and np.all(counts >= 1)
                    and np.all(np.isfinite(bad_rates))
                )

        def monotonic_compatible(
            previous_start: int,
            previous_end: int,
            current_start: int,
            current_end: int,
        ) -> bool:
            previous_br = segment_bad_rate[
                previous_start,
                previous_end,
            ]
            current_br = segment_bad_rate[
                current_start,
                current_end,
            ]

            if risk_direction == "higher_score_higher_risk":
                return bool(np.all(previous_br <= current_br))

            return bool(np.all(previous_br >= current_br))

        # Estado exato:
        # (start, end) identifica explicitamente o último grupo.
        #
        # layers[k][(start, end)] =
        #     melhor custo para particionar 0..end em k grupos,
        #     sendo [start, end] o último grupo.
        layers: dict[
            int,
            dict[tuple[int, int], tuple[float, float]],
        ] = {}

        parents: dict[
            int,
            dict[
                tuple[int, int],
                tuple[int, int] | None,
            ],
        ] = {}

        first_layer: dict[
            tuple[int, int],
            tuple[float, float],
        ] = {}

        first_parents: dict[
            tuple[int, int],
            tuple[int, int] | None,
        ] = {}

        for end in range(n_bins):
            if not feasible_segment[0, end]:
                continue

            state = (0, end)
            first_layer[state] = (
                segment_psi_cost[0, end],
                segment_br_cost[0, end],
            )
            first_parents[state] = None

        layers[1] = first_layer
        parents[1] = first_parents

        for group_index in range(2, n_groups + 1):
            previous_layer = layers[group_index - 1]

            # Indexa os estados anteriores pelo último pré-bin coberto.
            previous_states_by_end: dict[
                int,
                list[tuple[int, int]],
            ] = {}

            for state in previous_layer:
                previous_states_by_end.setdefault(
                    state[1],
                    [],
                ).append(state)

            current_layer: dict[
                tuple[int, int],
                tuple[float, float],
            ] = {}

            current_parents: dict[
                tuple[int, int],
                tuple[int, int] | None,
            ] = {}

            # O novo grupo começa em current_start.
            for current_start in range(1, n_bins):
                required_previous_end = current_start - 1

                compatible_previous_states = (
                    previous_states_by_end.get(
                        required_previous_end,
                        [],
                    )
                )

                if not compatible_previous_states:
                    continue

                for current_end in range(current_start, n_bins):
                    if not feasible_segment[
                        current_start,
                        current_end,
                    ]:
                        continue

                    current_state = (
                        current_start,
                        current_end,
                    )

                    best_cost: tuple[float, float] | None = None
                    best_parent: tuple[int, int] | None = None

                    current_segment_cost = (
                        segment_psi_cost[
                            current_start,
                            current_end,
                        ],
                        segment_br_cost[
                            current_start,
                            current_end,
                        ],
                    )

                    for previous_state in compatible_previous_states:
                        previous_start, previous_end = previous_state

                        if not monotonic_compatible(
                            previous_start=previous_start,
                            previous_end=previous_end,
                            current_start=current_start,
                            current_end=current_end,
                        ):
                            continue

                        previous_cost = previous_layer[
                            previous_state
                        ]

                        candidate_cost = (
                            previous_cost[0]
                            + current_segment_cost[0],
                            previous_cost[1]
                            + current_segment_cost[1],
                        )

                        if _lexicographic_better(
                            candidate=candidate_cost,
                            incumbent=best_cost,
                        ):
                            best_cost = candidate_cost
                            best_parent = previous_state

                    if best_cost is not None:
                        current_layer[current_state] = best_cost
                        current_parents[current_state] = best_parent

            layers[group_index] = current_layer
            parents[group_index] = current_parents

        final_candidates = {
            state: cost
            for state, cost in layers[n_groups].items()
            if state[1] == n_bins - 1
        }

        if not final_candidates:
            results[n_groups] = BinningSolution(
                n_groups=n_groups,
                cutpoints=None,
                population_psi=None,
                bad_rate_instability=None,
                feasible=False,
            )
            continue

        best_final_state: tuple[int, int] | None = None
        best_final_cost: tuple[float, float] | None = None

        for state, cost in final_candidates.items():
            if _lexicographic_better(
                candidate=cost,
                incumbent=best_final_cost,
            ):
                best_final_state = state
                best_final_cost = cost

        assert best_final_state is not None
        assert best_final_cost is not None

        states_reversed: list[tuple[int, int]] = []
        current_state: tuple[int, int] | None = best_final_state

        for group_index in range(n_groups, 0, -1):
            if current_state is None:
                raise RuntimeError(
                    "Falha inesperada durante a reconstrução da solução."
                )

            states_reversed.append(current_state)
            current_state = parents[group_index][current_state]

        states = list(reversed(states_reversed))

        cutpoint_indices = [
            segment_end
            for _, segment_end in states[:-1]
        ]

        cutpoints = [
            float(bin_boundaries[index])
            for index in cutpoint_indices
        ]

        results[n_groups] = BinningSolution(
            n_groups=n_groups,
            cutpoints=cutpoints,
            population_psi=float(best_final_cost[0]),
            bad_rate_instability=float(best_final_cost[1]),
            feasible=True,
        )

    return results

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Iterable, Literal, Optional, Sequence, Tuple, Union

import numpy as np
import pandas as pd


Trend = Literal["auto", "increasing", "decreasing"]


@dataclass(frozen=True)
class BinningSolution:
    """One candidate solution for a fixed number of final groups."""

    n_groups: int
    cutpoints: Tuple[float, ...]
    segments: Tuple[Tuple[int, int], ...]
    objective: float
    fit_cost: float
    psi_cost: float
    separation_cost: float
    complexity_cost: float
    min_overall_share: float
    min_monthly_share: float
    min_adjacent_gap: float
    group_summary: pd.DataFrame
    monthly_summary: pd.DataFrame


# ---------------------------------------------------------------------------
# Validation and weighted utilities
# ---------------------------------------------------------------------------


def _weighted_quantile(
    values: np.ndarray,
    quantiles: np.ndarray,
    sample_weight: np.ndarray,
) -> np.ndarray:
    """Return weighted quantiles using midpoint cumulative probabilities."""
    order = np.argsort(values, kind="mergesort")
    x = np.asarray(values, dtype=float)[order]
    w = np.asarray(sample_weight, dtype=float)[order]

    positive = w > 0
    x, w = x[positive], w[positive]
    if x.size == 0 or w.sum() <= 0:
        raise ValueError("At least one strictly positive weight is required.")

    cumulative = np.cumsum(w) - 0.5 * w
    cumulative /= w.sum()
    return np.interp(quantiles, cumulative, x, left=x[0], right=x[-1])


def _kish_effective_n(weights: np.ndarray) -> float:
    w = np.asarray(weights, dtype=float)
    numerator = float(w.sum() ** 2)
    denominator = float(np.square(w).sum())
    return numerator / denominator if denominator > 0 else 0.0


def _weighted_corr(x: np.ndarray, y: np.ndarray, w: np.ndarray) -> float:
    total = w.sum()
    if total <= 0:
        return 0.0
    mx = np.sum(w * x) / total
    my = np.sum(w * y) / total
    vx = np.sum(w * np.square(x - mx)) / total
    vy = np.sum(w * np.square(y - my)) / total
    if vx <= 0 or vy <= 0:
        return 0.0
    cov = np.sum(w * (x - mx) * (y - my)) / total
    return float(cov / np.sqrt(vx * vy))


def _pava_projection(y: np.ndarray, increasing: bool = True) -> np.ndarray:
    """Euclidean projection onto a monotone cone using the PAVA algorithm."""
    values = np.asarray(y, dtype=float)
    if values.ndim != 1:
        raise ValueError("PAVA input must be one-dimensional.")
    if not increasing:
        return -_pava_projection(-values, increasing=True)

    block_values: list[float] = []
    block_weights: list[float] = []
    block_starts: list[int] = []
    block_ends: list[int] = []

    for idx, value in enumerate(values):
        block_values.append(float(value))
        block_weights.append(1.0)
        block_starts.append(idx)
        block_ends.append(idx)

        while (
            len(block_values) >= 2
            and block_values[-2] > block_values[-1]
        ):
            w_left, w_right = block_weights[-2], block_weights[-1]
            merged_weight = w_left + w_right
            merged_value = (
                w_left * block_values[-2] + w_right * block_values[-1]
            ) / merged_weight
            merged_start = block_starts[-2]
            merged_end = block_ends[-1]

            block_values[-2:] = [merged_value]
            block_weights[-2:] = [merged_weight]
            block_starts[-2:] = [merged_start]
            block_ends[-2:] = [merged_end]

    result = np.empty_like(values, dtype=float)
    for value, start, end in zip(block_values, block_starts, block_ends):
        result[start : end + 1] = value
    return result


def _temporal_laplacian(n_periods: int) -> np.ndarray:
    """Return D' D for first differences over equally spaced periods."""
    if n_periods <= 1:
        return np.zeros((n_periods, n_periods), dtype=float)
    lap = np.zeros((n_periods, n_periods), dtype=float)
    diag = np.full(n_periods, 2.0)
    diag[0] = diag[-1] = 1.0
    np.fill_diagonal(lap, diag)
    off = np.full(n_periods - 1, -1.0)
    lap[np.arange(n_periods - 1), np.arange(1, n_periods)] = off
    lap[np.arange(1, n_periods), np.arange(n_periods - 1)] = off
    return lap


# ---------------------------------------------------------------------------
# Convex isotonic + temporal smoothing stage
# ---------------------------------------------------------------------------


def _fit_isotonic_temporal_surface(
    observed_rates: np.ndarray,
    cell_weights: np.ndarray,
    *,
    increasing: bool,
    lambda_time: float,
    admm_rho: float,
    admm_max_iter: int,
    admm_tol: float,
) -> Tuple[np.ndarray, Dict[str, Any]]:
    """
    Solve, with ADMM:

      min_Theta  1/2 sum_bt W_bt (R_bt - Theta_bt)^2
                 + lambda_time/2 sum_b,t>1 (Theta_bt - Theta_b,t-1)^2
      subject to Theta_1t <= ... <= Theta_Bt (or the decreasing analogue).

    Missing cells have W_bt = 0 and are inferred from temporal regularization and
    the monotonic projection.
    """
    rates = np.asarray(observed_rates, dtype=float)
    weights = np.asarray(cell_weights, dtype=float)
    if rates.shape != weights.shape:
        raise ValueError("observed_rates and cell_weights must have equal shapes.")

    n_bins, n_periods = rates.shape
    positive = weights > 0
    if not positive.any():
        raise ValueError("No positive aggregate cell weight was found.")

    # Normalize reliability weights so lambda_time and rho are not tied to the
    # arbitrary expansion scale of population weights.
    scale = float(np.mean(weights[positive]))
    w_norm = weights / scale

    global_rate = float(np.sum(weights[positive] * rates[positive]) / weights[positive].sum())
    filled = rates.copy()
    for b in range(n_bins):
        mask = positive[b]
        if mask.any():
            row_mean = float(np.sum(weights[b, mask] * rates[b, mask]) / weights[b, mask].sum())
        else:
            row_mean = global_rate
        filled[b, ~mask] = row_mean

    theta = filled.copy()
    z = np.column_stack(
        [_pava_projection(theta[:, t], increasing=increasing) for t in range(n_periods)]
    )
    u = np.zeros_like(theta)
    temporal_penalty = _temporal_laplacian(n_periods)

    primal_residual = np.inf
    dual_residual = np.inf
    iterations = 0

    for iterations in range(1, admm_max_iter + 1):
        # Theta update separates by score pre-bin and couples periods.
        for b in range(n_bins):
            matrix = (
                np.diag(w_norm[b])
                + lambda_time * temporal_penalty
                + admm_rho * np.eye(n_periods)
            )
            rhs = w_norm[b] * filled[b] + admm_rho * (z[b] - u[b])
            theta[b] = np.linalg.solve(matrix, rhs)

        previous_z = z.copy()

        # Z update is the exact Euclidean projection onto the monotone cone,
        # independently for each period.
        shifted = theta + u
        for t in range(n_periods):
            z[:, t] = _pava_projection(shifted[:, t], increasing=increasing)

        u += theta - z

        primal_residual = float(np.linalg.norm(theta - z))
        dual_residual = float(admm_rho * np.linalg.norm(z - previous_z))
        scale_norm = max(1.0, float(np.linalg.norm(theta)), float(np.linalg.norm(z)))
        if max(primal_residual, dual_residual) <= admm_tol * scale_norm:
            break

    diagnostics = {
        "iterations": iterations,
        "converged": iterations < admm_max_iter,
        "primal_residual": primal_residual,
        "dual_residual": dual_residual,
        "weight_normalization": scale,
    }
    return z, diagnostics


# ---------------------------------------------------------------------------
# Segment and transition costs
# ---------------------------------------------------------------------------


def _precompute_segment_statistics(
    smoothed_rates: np.ndarray,
    population_weights: np.ndarray,
    *,
    min_overall_share: float,
    min_monthly_share: float,
    psi_epsilon: float,
) -> Dict[str, np.ndarray]:
    n_bins, n_periods = smoothed_rates.shape
    total_weight = float(population_weights.sum())
    month_totals = population_weights.sum(axis=0)
    month_importance = month_totals / month_totals.sum()

    shape2 = (n_bins, n_bins)
    fit = np.full(shape2, np.inf)
    psi = np.full(shape2, np.inf)
    valid = np.zeros(shape2, dtype=bool)
    total_share = np.full(shape2, np.nan)
    minimum_month_share = np.full(shape2, np.nan)
    monthly_means = np.full((n_bins, n_bins, n_periods), np.nan)
    monthly_segment_weights = np.zeros((n_bins, n_bins, n_periods))

    # Prefix sums make all segment aggregates O(T).
    prefix_w = np.vstack([np.zeros(n_periods), np.cumsum(population_weights, axis=0)])
    prefix_wr = np.vstack(
        [np.zeros(n_periods), np.cumsum(population_weights * smoothed_rates, axis=0)]
    )
    prefix_wr2 = np.vstack(
        [np.zeros(n_periods), np.cumsum(population_weights * np.square(smoothed_rates), axis=0)]
    )

    for left in range(n_bins):
        for right in range(left, n_bins):
            seg_w_t = prefix_w[right + 1] - prefix_w[left]
            seg_wr_t = prefix_wr[right + 1] - prefix_wr[left]
            seg_wr2_t = prefix_wr2[right + 1] - prefix_wr2[left]
            monthly_segment_weights[left, right] = seg_w_t

            with np.errstate(divide="ignore", invalid="ignore"):
                seg_mean_t = np.divide(
                    seg_wr_t,
                    seg_w_t,
                    out=np.full(n_periods, np.nan),
                    where=seg_w_t > 0,
                )
            monthly_means[left, right] = seg_mean_t

            overall_share = float(seg_w_t.sum() / total_weight)
            month_share = np.divide(
                seg_w_t,
                month_totals,
                out=np.zeros_like(seg_w_t),
                where=month_totals > 0,
            )
            min_month_share_value = float(month_share.min())
            total_share[left, right] = overall_share
            minimum_month_share[left, right] = min_month_share_value

            is_valid = (
                overall_share >= min_overall_share
                and min_month_share_value >= min_monthly_share
                and np.all(seg_w_t > 0)
            )
            valid[left, right] = is_valid
            if not is_valid:
                continue

            # Weighted within-segment loss after replacing the pre-bin surface by
            # one monthly rate per final group. It is normalized to rate^2 units.
            sse_t = seg_wr2_t - np.divide(
                np.square(seg_wr_t),
                seg_w_t,
                out=np.zeros_like(seg_wr_t),
                where=seg_w_t > 0,
            )
            fit[left, right] = float(sse_t.sum() / total_weight)

            # PSI contribution is additive across final groups. The reference
            # share is the population-weighted share over all periods.
            reference_share = overall_share
            p = np.clip(month_share, psi_epsilon, None)
            q = max(reference_share, psi_epsilon)
            psi_t = (p - q) * np.log(p / q)
            psi[left, right] = float(np.sum(month_importance * psi_t))

    return {
        "fit": fit,
        "psi": psi,
        "valid": valid,
        "total_share": total_share,
        "minimum_month_share": minimum_month_share,
        "monthly_means": monthly_means,
        "monthly_weights": monthly_segment_weights,
        "month_importance": month_importance,
    }


def _precompute_transition_costs(
    monthly_means: np.ndarray,
    valid: np.ndarray,
    month_importance: np.ndarray,
    *,
    increasing: bool,
    min_rate_diff: float,
    monotonic_tolerance: float,
) -> Tuple[np.ndarray, np.ndarray]:
    """
    transition[left_previous, boundary, right_current] compares
    previous=[left_previous, boundary] with current=[boundary+1, right_current].
    """
    n_bins = valid.shape[0]
    transition = np.full((n_bins, n_bins, n_bins), np.inf)
    min_gap = np.full((n_bins, n_bins, n_bins), np.nan)

    for previous_left in range(n_bins):
        for boundary in range(previous_left, n_bins - 1):
            if not valid[previous_left, boundary]:
                continue
            previous_mean = monthly_means[previous_left, boundary]
            for current_right in range(boundary + 1, n_bins):
                current_left = boundary + 1
                if not valid[current_left, current_right]:
                    continue
                current_mean = monthly_means[current_left, current_right]
                gap = current_mean - previous_mean
                if not increasing:
                    gap = -gap

                if np.any(gap < -monotonic_tolerance):
                    continue  # hard no-inversion rule

                shortfall = np.maximum(0.0, min_rate_diff - gap)
                transition[previous_left, boundary, current_right] = float(
                    np.sum(month_importance * np.square(shortfall))
                )
                min_gap[previous_left, boundary, current_right] = float(np.min(gap))

    return transition, min_gap


# ---------------------------------------------------------------------------
# Exact dynamic program with the last segment retained in the state
# ---------------------------------------------------------------------------


def _solve_all_group_counts(
    segment_cost: np.ndarray,
    valid: np.ndarray,
    transition_cost: np.ndarray,
    *,
    min_groups: int,
    max_groups: int,
) -> Dict[int, Dict[str, Any]]:
    """
    Exact first-order segmentation DP.

    dp[k, a, b] is the minimum cost for partitioning pre-bins 0..b into k
    groups when the last group is exactly [a, b]. Retaining both a and b keeps
    enough state for an adjacent-group transition cost, avoiding the invalid
    state compression dp[k, b] used in simpler implementations.
    """
    n_bins = segment_cost.shape[0]
    max_groups = min(max_groups, n_bins)
    dp = np.full((max_groups + 1, n_bins, n_bins), np.inf)
    predecessor = np.full((max_groups + 1, n_bins, n_bins), -1, dtype=int)

    for end in range(n_bins):
        if valid[0, end]:
            dp[1, 0, end] = segment_cost[0, end]

    for k in range(2, max_groups + 1):
        # Last group [start, end] needs at least k-1 preceding pre-bins/groups.
        for end in range(k - 1, n_bins):
            for start in range(k - 1, end + 1):
                if not valid[start, end]:
                    continue
                boundary = start - 1
                best_value = np.inf
                best_previous_start = -1

                # Previous group is [previous_start, boundary].
                for previous_start in range(k - 2, boundary + 1):
                    previous_value = dp[k - 1, previous_start, boundary]
                    if not np.isfinite(previous_value):
                        continue
                    trans = transition_cost[previous_start, boundary, end]
                    if not np.isfinite(trans):
                        continue
                    candidate = previous_value + trans + segment_cost[start, end]
                    if candidate < best_value:
                        best_value = candidate
                        best_previous_start = previous_start

                if best_previous_start >= 0:
                    dp[k, start, end] = best_value
                    predecessor[k, start, end] = best_previous_start

    solutions: Dict[int, Dict[str, Any]] = {}
    final_end = n_bins - 1
    for k in range(min_groups, max_groups + 1):
        final_start = int(np.argmin(dp[k, :, final_end]))
        value = float(dp[k, final_start, final_end])
        if not np.isfinite(value):
            continue

        segments: list[Tuple[int, int]] = []
        start, end = final_start, final_end
        current_k = k
        while current_k >= 1:
            segments.append((start, end))
            if current_k == 1:
                break
            previous_start = int(predecessor[current_k, start, end])
            end = start - 1
            start = previous_start
            current_k -= 1
        segments.reverse()
        solutions[k] = {"segments": tuple(segments), "dp_cost": value}

    return solutions


# ---------------------------------------------------------------------------
# Public API
# ---------------------------------------------------------------------------


def isotonic_temporal_binning_dp(
    df: pd.DataFrame,
    *,
    score_col: str = "score",
    time_col: str = "month",
    target_col: str = "target",
    weight_col: Optional[str] = None,
    n_prebins: int = 50,
    min_groups: int = 3,
    max_groups: int = 10,
    monotonic_trend: Trend = "auto",
    lambda_time: float = 2.0,
    psi_strength: float = 0.05,
    separation_strength: float = 1.0,
    min_rate_diff: float = 0.005,
    complexity_strength: float = 1.0,
    min_group_weight_share: float = 0.03,
    min_monthly_group_weight_share: float = 0.01,
    objective_tolerance: float = 0.05,
    top_n_suggestions: int = 3,
    psi_epsilon: float = 1e-8,
    monotonic_tolerance: float = 1e-10,
    admm_rho: float = 1.0,
    admm_max_iter: int = 2_000,
    admm_tol: float = 1e-7,
    time_order: Optional[Sequence[Any]] = None,
) -> Dict[str, Any]:
    """
    Create temporally stable, monotone score groups with an exact segmentation DP.

    Parameters
    ----------
    df:
        Input data with one row per observation.
    score_col, time_col, target_col:
        Score, ordered time reference, and binary target columns.
    weight_col:
        Optional non-negative sampling/population weight. With None, every row has
        weight 1. Weights affect quantiles, rates, support, isotonic reliability,
        PSI, segment losses, summaries, and cutpoint selection.
    n_prebins:
        Requested number of weighted-quantile pre-bins. Ties may reduce it.
    min_groups, max_groups:
        Candidate range for the final number of groups.
    monotonic_trend:
        "increasing", "decreasing", or "auto" from weighted score-target
        correlation.
    lambda_time:
        Temporal roughness penalty in the convex smoothing stage.
    psi_strength:
        Multiplier for additive weighted PSI. Internally it is multiplied by the
        weighted target variance to place it in rate-squared units.
    separation_strength:
        Multiplier for the adjacent-group minimum-gap shortfall penalty.
    min_rate_diff:
        Desired minimum bad-rate difference between adjacent groups in every
        period. Violations are penalized quadratically; inversions are forbidden.
    complexity_strength:
        Multiplier of a BIC-like per-group penalty based on Kish effective n.
    min_group_weight_share, min_monthly_group_weight_share:
        Hard population-weight support constraints overall and within every period.
    objective_tolerance:
        Relative tolerance used to produce the "simpler" and "more granular"
        recommendations around the minimum objective.

    Returns
    -------
    dict with:
      - cutpoints: best finite score cutpoints
      - best_solution: BinningSolution
      - suggestions: ranked DataFrame, normally containing three recommendations
      - solutions: all feasible BinningSolution objects keyed by group count
      - prebin_table, observed_rates, smoothed_rates
      - smoothing_diagnostics and configuration
    """
    required = {score_col, time_col, target_col}
    if weight_col is not None:
        required.add(weight_col)
    missing = required.difference(df.columns)
    if missing:
        raise KeyError(f"Missing required columns: {sorted(missing)}")

    columns = [score_col, time_col, target_col]
    if weight_col is not None:
        columns.append(weight_col)
    work = df[columns].copy()
    work = work.dropna(subset=[score_col, time_col, target_col])

    if weight_col is None:
        internal_weight_col = "__weight__"
        work[internal_weight_col] = 1.0
    else:
        internal_weight_col = weight_col
        work = work.dropna(subset=[weight_col])
        work[internal_weight_col] = pd.to_numeric(work[weight_col], errors="raise")

    work[score_col] = pd.to_numeric(work[score_col], errors="raise")
    work[target_col] = pd.to_numeric(work[target_col], errors="raise")

    if work.empty:
        raise ValueError("No complete observations remain after dropping missing values.")
    if (~np.isfinite(work[score_col])).any():
        raise ValueError("score_col must contain finite values.")
    if (~np.isfinite(work[target_col])).any():
        raise ValueError("target_col must contain finite values.")
    if (~np.isfinite(work[internal_weight_col])).any():
        raise ValueError("Weights must be finite.")
    if (work[internal_weight_col] < 0).any():
        raise ValueError("Weights must be non-negative.")
    work = work.loc[work[internal_weight_col] > 0].copy()
    if work.empty:
        raise ValueError("At least one observation must have positive weight.")
    if not work[target_col].between(0, 1).all():
        raise ValueError("target_col must be binary or a fractional event indicator in [0, 1].")

    if not (2 <= n_prebins):
        raise ValueError("n_prebins must be at least 2.")
    if not (1 <= min_groups <= max_groups):
        raise ValueError("Require 1 <= min_groups <= max_groups.")
    for name, value in {
        "lambda_time": lambda_time,
        "psi_strength": psi_strength,
        "separation_strength": separation_strength,
        "min_rate_diff": min_rate_diff,
        "complexity_strength": complexity_strength,
        "min_group_weight_share": min_group_weight_share,
        "min_monthly_group_weight_share": min_monthly_group_weight_share,
    }.items():
        if value < 0:
            raise ValueError(f"{name} must be non-negative.")

    if time_order is None:
        try:
            periods = list(pd.Index(work[time_col].unique()).sort_values())
        except TypeError as exc:
            raise TypeError(
                "time_col values are not mutually sortable; provide time_order explicitly."
            ) from exc
    else:
        periods = list(time_order)
        observed_periods = set(work[time_col].unique())
        if set(periods) != observed_periods or len(periods) != len(observed_periods):
            raise ValueError("time_order must contain every observed period exactly once.")

    period_to_index = {period: idx for idx, period in enumerate(periods)}
    work["__time_index__"] = work[time_col].map(period_to_index)

    x = work[score_col].to_numpy(dtype=float)
    y = work[target_col].to_numpy(dtype=float)
    w = work[internal_weight_col].to_numpy(dtype=float)

    if monotonic_trend == "auto":
        increasing = _weighted_corr(x, y, w) >= 0
        resolved_trend = "increasing" if increasing else "decreasing"
    elif monotonic_trend in {"increasing", "decreasing"}:
        increasing = monotonic_trend == "increasing"
        resolved_trend = monotonic_trend
    else:
        raise ValueError("monotonic_trend must be 'auto', 'increasing', or 'decreasing'.")

    quantiles = np.linspace(0.0, 1.0, n_prebins + 1)[1:-1]
    raw_internal_edges = _weighted_quantile(x, quantiles, w)
    internal_edges = np.unique(raw_internal_edges)
    internal_edges = internal_edges[(internal_edges > np.min(x)) & (internal_edges < np.max(x))]
    edges = np.concatenate(([-np.inf], internal_edges, [np.inf]))

    work["__prebin__"] = pd.cut(
        work[score_col], bins=edges, labels=False, include_lowest=True, right=True
    ).astype(int)
    actual_prebins = int(work["__prebin__"].nunique())
    if actual_prebins < max_groups:
        max_groups = actual_prebins
    if actual_prebins < min_groups:
        raise ValueError(
            f"Only {actual_prebins} distinct weighted pre-bins were created, fewer than "
            f"min_groups={min_groups}. Reduce min_groups or inspect score ties."
        )

    # pd.cut may theoretically leave empty bins after tied quantiles. Compress them
    # while preserving score order and rebuild the finite boundary vector.
    occupied = sorted(work["__prebin__"].unique())
    remap = {old: new for new, old in enumerate(occupied)}
    work["__prebin__"] = work["__prebin__"].map(remap).astype(int)
    actual_prebins = len(occupied)

    # Boundary after compressed pre-bin i is the maximum score assigned to it.
    # This remains valid even when some requested quantile bins were empty.
    prebin_max_score = (
        work.groupby("__prebin__", observed=True)[score_col].max().sort_index().to_numpy()
    )

    work["__weighted_target__"] = work[internal_weight_col] * work[target_col]
    aggregate = (
        work.groupby(["__prebin__", "__time_index__"], observed=True)
        .agg(
            population_weight=(internal_weight_col, "sum"),
            weighted_events=("__weighted_target__", "sum"),
            raw_rows=(target_col, "size"),
        )
        .reset_index()
        .rename(columns={"__prebin__": "prebin_index", "__time_index__": "time_index"})
    )
    aggregate["observed_rate"] = (
        aggregate["weighted_events"] / aggregate["population_weight"]
    )

    shape = (actual_prebins, len(periods))
    population_weights = np.zeros(shape, dtype=float)
    weighted_events = np.zeros(shape, dtype=float)
    observed_rates = np.full(shape, np.nan, dtype=float)
    raw_rows = np.zeros(shape, dtype=int)
    for row in aggregate.itertuples(index=False):
        b = int(row.prebin_index)
        t = int(row.time_index)
        population_weights[b, t] = float(row.population_weight)
        weighted_events[b, t] = float(row.weighted_events)
        observed_rates[b, t] = float(row.observed_rate)
        raw_rows[b, t] = int(row.raw_rows)

    smoothed_rates, smoothing_diagnostics = _fit_isotonic_temporal_surface(
        observed_rates,
        population_weights,
        increasing=increasing,
        lambda_time=lambda_time,
        admm_rho=admm_rho,
        admm_max_iter=admm_max_iter,
        admm_tol=admm_tol,
    )

    segment_stats = _precompute_segment_statistics(
        smoothed_rates,
        population_weights,
        min_overall_share=min_group_weight_share,
        min_monthly_share=min_monthly_group_weight_share,
        psi_epsilon=psi_epsilon,
    )
    transition_base, transition_min_gap = _precompute_transition_costs(
        segment_stats["monthly_means"],
        segment_stats["valid"],
        segment_stats["month_importance"],
        increasing=increasing,
        min_rate_diff=min_rate_diff,
        monotonic_tolerance=monotonic_tolerance,
    )

    weighted_target_mean = float(np.sum(w * y) / w.sum())
    target_variance = max(weighted_target_mean * (1.0 - weighted_target_mean), 1e-12)
    n_eff = max(_kish_effective_n(w), 2.0)
    psi_multiplier = psi_strength * target_variance
    separation_multiplier = separation_strength
    complexity_per_group = (
        complexity_strength * target_variance * np.log(n_eff) / n_eff
    )

    segment_cost = segment_stats["fit"] + psi_multiplier * segment_stats["psi"]
    transition_cost = separation_multiplier * transition_base

    raw_solutions = _solve_all_group_counts(
        segment_cost,
        segment_stats["valid"],
        transition_cost,
        min_groups=min_groups,
        max_groups=max_groups,
    )
    if not raw_solutions:
        raise ValueError(
            "No feasible partition was found. Reduce the minimum group shares, "
            "reduce min_groups, or increase n_prebins/data coverage."
        )

    solutions: Dict[int, BinningSolution] = {}
    for k, raw_solution in raw_solutions.items():
        segments = raw_solution["segments"]
        fit_cost = float(sum(segment_stats["fit"][a, b] for a, b in segments))
        psi_cost_unscaled = float(sum(segment_stats["psi"][a, b] for a, b in segments))
        psi_cost = psi_multiplier * psi_cost_unscaled

        separation_cost = 0.0
        transition_gaps: list[float] = []
        for previous, current in zip(segments[:-1], segments[1:]):
            previous_left, boundary = previous
            current_left, current_right = current
            if current_left != boundary + 1:
                raise RuntimeError("Internal segmentation continuity error.")
            separation_cost += transition_cost[previous_left, boundary, current_right]
            transition_gaps.append(
                transition_min_gap[previous_left, boundary, current_right]
            )

        complexity_cost = complexity_per_group * k
        objective = fit_cost + psi_cost + separation_cost + complexity_cost

        cutpoints = tuple(float(prebin_max_score[b]) for _, b in segments[:-1])
        group_records = []
        monthly_records = []
        all_weight = population_weights.sum()
        month_totals = population_weights.sum(axis=0)

        for group_id, (left, right) in enumerate(segments, start=1):
            group_weight_t = segment_stats["monthly_weights"][left, right]
            group_rate_t = segment_stats["monthly_means"][left, right]
            overall_weight = float(group_weight_t.sum())
            overall_rate = float(
                np.sum(group_weight_t * group_rate_t) / overall_weight
            )
            raw_events_t = weighted_events[left : right + 1].sum(axis=0)
            raw_rows_t = raw_rows[left : right + 1].sum(axis=0)
            raw_rate_t = np.divide(
                raw_events_t,
                group_weight_t,
                out=np.full_like(raw_events_t, np.nan, dtype=float),
                where=group_weight_t > 0,
            )
            raw_overall_rate = float(raw_events_t.sum() / overall_weight)
            group_records.append(
                {
                    "group": group_id,
                    "prebin_start": left,
                    "prebin_end": right,
                    "score_lower_exclusive": -np.inf if group_id == 1 else cutpoints[group_id - 2],
                    "score_upper_inclusive": np.inf if group_id == k else cutpoints[group_id - 1],
                    "population_weight": overall_weight,
                    "population_share": overall_weight / all_weight,
                    "raw_rows": int(raw_rows_t.sum()),
                    "observed_weighted_bad_rate": raw_overall_rate,
                    "smoothed_bad_rate": overall_rate,
                }
            )
            for t, period in enumerate(periods):
                monthly_records.append(
                    {
                        "group": group_id,
                        time_col: period,
                        "population_weight": float(group_weight_t[t]),
                        "population_share": float(group_weight_t[t] / month_totals[t]),
                        "raw_rows": int(raw_rows_t[t]),
                        "observed_weighted_bad_rate": float(raw_rate_t[t]),
                        "smoothed_bad_rate": float(group_rate_t[t]),
                    }
                )

        group_summary = pd.DataFrame(group_records)
        monthly_summary = pd.DataFrame(monthly_records)
        min_overall = float(group_summary["population_share"].min())
        min_monthly = float(monthly_summary["population_share"].min())
        minimum_gap = float(min(transition_gaps)) if transition_gaps else np.inf

        solutions[k] = BinningSolution(
            n_groups=k,
            cutpoints=cutpoints,
            segments=segments,
            objective=float(objective),
            fit_cost=fit_cost,
            psi_cost=float(psi_cost),
            separation_cost=float(separation_cost),
            complexity_cost=float(complexity_cost),
            min_overall_share=min_overall,
            min_monthly_share=min_monthly,
            min_adjacent_gap=minimum_gap,
            group_summary=group_summary,
            monthly_summary=monthly_summary,
        )

    ordered = sorted(solutions.values(), key=lambda solution: solution.objective)
    best = ordered[0]
    threshold = best.objective * (1.0 + objective_tolerance)
    near_optimal = [solution for solution in ordered if solution.objective <= threshold]

    selected: list[Tuple[str, BinningSolution]] = [("melhor_objetivo", best)]
    if near_optimal:
        simplest = min(near_optimal, key=lambda solution: solution.n_groups)
        most_granular = max(near_optimal, key=lambda solution: solution.n_groups)
        for label, candidate in [
            ("mais_parcimoniosa", simplest),
            ("mais_granular", most_granular),
        ]:
            if all(candidate.n_groups != existing.n_groups for _, existing in selected):
                selected.append((label, candidate))

    for candidate in ordered:
        if len(selected) >= top_n_suggestions:
            break
        if all(candidate.n_groups != existing.n_groups for _, existing in selected):
            selected.append(("alternativa", candidate))

    suggestion_rows = []
    for rank, (label, solution) in enumerate(selected[:top_n_suggestions], start=1):
        suggestion_rows.append(
            {
                "rank": rank,
                "recommendation": label,
                "n_groups": solution.n_groups,
                "objective": solution.objective,
                "fit_cost": solution.fit_cost,
                "psi_cost": solution.psi_cost,
                "separation_cost": solution.separation_cost,
                "complexity_cost": solution.complexity_cost,
                "min_overall_share": solution.min_overall_share,
                "min_monthly_share": solution.min_monthly_share,
                "min_adjacent_gap": solution.min_adjacent_gap,
                "cutpoints": solution.cutpoints,
            }
        )
    suggestions = pd.DataFrame(suggestion_rows)

    prebin_records = []
    for b in range(actual_prebins):
        mask = work["__prebin__"].to_numpy() == b
        wb = w[mask]
        yb = y[mask]
        prebin_records.append(
            {
                "prebin": b,
                "score_min": float(np.min(x[mask])),
                "score_max": float(np.max(x[mask])),
                "population_weight": float(wb.sum()),
                "population_share": float(wb.sum() / w.sum()),
                "weighted_bad_rate": float(np.sum(wb * yb) / wb.sum()),
                "raw_rows": int(mask.sum()),
            }
        )
    prebin_table = pd.DataFrame(prebin_records)

    observed_rates_df = pd.DataFrame(observed_rates, index=range(actual_prebins), columns=periods)
    smoothed_rates_df = pd.DataFrame(smoothed_rates, index=range(actual_prebins), columns=periods)
    population_weights_df = pd.DataFrame(
        population_weights, index=range(actual_prebins), columns=periods
    )
    weighted_events_df = pd.DataFrame(
        weighted_events, index=range(actual_prebins), columns=periods
    )
    observed_rates_df.index.name = smoothed_rates_df.index.name = "prebin"
    population_weights_df.index.name = "prebin"
    weighted_events_df.index.name = "prebin"

    return {
        "cutpoints": best.cutpoints,
        "best_solution": best,
        "suggestions": suggestions,
        "solutions": solutions,
        "prebin_table": prebin_table,
        "observed_rates": observed_rates_df,
        "smoothed_rates": smoothed_rates_df,
        "population_weights": population_weights_df,
        "weighted_events": weighted_events_df,
        "smoothing_diagnostics": smoothing_diagnostics,
        "configuration": {
            "resolved_monotonic_trend": resolved_trend,
            "requested_prebins": n_prebins,
            "actual_prebins": actual_prebins,
            "periods": tuple(periods),
            "weighted_target_mean": weighted_target_mean,
            "weighted_target_variance": target_variance,
            "kish_effective_n": n_eff,
            "psi_multiplier": psi_multiplier,
            "separation_multiplier": separation_multiplier,
            "complexity_per_group": complexity_per_group,
        },
    }


def apply_isotonic_temporal_binning(
    scores: Union[pd.Series, Iterable[float], np.ndarray],
    cutpoints: Sequence[float],
    *,
    labels_start_at_one: bool = True,
) -> np.ndarray:
    """Apply learned cutpoints to new scores using right-closed intervals."""
    values = np.asarray(scores, dtype=float)
    cuts = np.asarray(sorted(cutpoints), dtype=float)
    groups = np.searchsorted(cuts, values, side="left")
    if labels_start_at_one:
        groups = groups + 1
    return groups